# HookLink2iOS — Ocarina of Time (Switch 2 Edition)

**Notebook completo: URL → NUP → Google Drive → preparação Linux → GitHub Actions macOS-26 → IPA de volta ao Drive.**

Esta versão corrige o erro anterior: **não existe mais a seleção `Origem [drive/url]`**. Cole diretamente a URL HTTPS completa quando solicitada. O nome e a extensão são identificados pelo `Content-Disposition`/URL. Não é preciso digitar `url`, `drive` ou um nome de arquivo.

## Fluxo

1. O Colab recebe a URL HTTPS que você fornecer. A URL assinada fica somente na memória do runtime e não é gravada no repositório nem exibida nos logs.
2. Antes de acessar o endereço externo, o notebook procura um NUP validado em `HookLink2iOS/Inputs` no Drive. A identidade do link ignora os parâmetros temporários de assinatura, então um link renovado para o mesmo caminho reutiliza o cache.
3. Na primeira execução, baixa o arquivo. Se for `.rar`, `.zip`, `.7z` ou tar compactado, inspeciona o arquivo, recusa caminhos inseguros/links, extrai exatamente um `.nup`, envia o NUP ao Drive e verifica tamanho/MD5. Só depois disso remove o arquivo compactado do Colab. O NUP extraído permanece no Drive.
4. Linux valida Title ID `0400486024DEC000` e versão `v131072`, gera `RomFS.img` e o overlay de resolução e envia os resultados para uma pasta específica da preparação.
5. Publica o handoff somente na branch `HookLink2iOS`; o workflow usa `macos-26`, nxnative fixado e build nativo iOS com `--romfs separate`.
6. O Colab aguarda o workflow do commit exato, baixa apenas o artifact `HookLink2iOS-Ocarina-<run>`, verifica o digest do ZIP e o SHA/tamanho da IPA e envia IPA, overlay e relatório para uma subpasta exclusiva daquela tentativa no Drive.

Se a mesma origem já tiver handoff e preparação válidos, o notebook reutiliza NUP/RomFS/overlay e publica apenas um novo identificador de build. Assim, é possível recompilar o alvo iOS sem baixar ou preparar o pacote novamente.

## Reutilização e autorização

- Usa a pasta raiz do notebook Link2iOS original: `19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm`. Não pergunta outro ID.
- O Colab usa o OAuth da sua conta Google para ler/gravar na pasta compartilhada. O runner macOS reutiliza o secret permanente `GOOGLE_SERVICE_ACCOUNT_JSON` já usado pelo workflow Link2iOS. Não é preciso criar outra conta de serviço, adicionar permissão `Secrets` ao PAT nem enviar o OAuth do Colab ao GitHub.
- Ao reabrir o notebook com a mesma URL, tenta recuperar o handoff da branch e os assets já salvos no Drive. Se encontrados e íntegros, não repete download, extração nem preparação Linux.
- O arquivo NUP fica em `HookLink2iOS/Inputs`; RomFS, overlay, IPA e relatório ficam na pasta da execução.
- PAT Fine-grained restrito a este repositório: `Contents: Read and write` e `Actions: Read`, no mesmo padrão operacional do notebook Link2iOS. Não é necessária permissão `Secrets` no PAT.
- Link assinado pode expirar. Nesse caso, gere outro e cole-o na célula de URL; o fingerprint não depende da assinatura.

**Alvo:** iPhone 17 Pro, GPU preset `g18p`; build nativo iOS, sem emulação.

## 0. Preparar o runtime

Instala as ferramentas de extração e clona somente a branch `HookLink2iOS`. O diretório de trabalho não é apagado ao repetir esta célula, preservando temporários da sessão. O cache durável principal permanece no Drive.

In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq libarchive-tools
%pip install -q backports.zstd
# Preservar as versões que já vêm no runtime Colab; não atualizar dependências globais.
import importlib.util
assert importlib.util.find_spec("requests") is not None, "requests ausente no runtime Colab."
assert importlib.util.find_spec("google.auth") is not None, "google-auth ausente no runtime Colab."
assert importlib.util.find_spec("googleapiclient") is not None, "google-api-python-client ausente no runtime Colab."
import requests, google.auth
from googleapiclient.discovery import build
print("requests:", requests.__version__)
print("google-auth:", getattr(google.auth, "__version__", "presente"))
print("Google API Client: disponível")


from pathlib import Path
import subprocess, sys, hashlib, json, os

REPO_DIR = Path("/content/HookLink2iOS-repo")
if not (REPO_DIR / ".git").is_dir():
    !git clone --depth 1 --branch HookLink2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/HookLink2iOS-repo
else:
    !git -C /content/HookLink2iOS-repo fetch --depth 1 origin HookLink2iOS
    !git -C /content/HookLink2iOS-repo checkout HookLink2iOS
    !git -C /content/HookLink2iOS-repo reset --hard FETCH_HEAD

REPO_HEAD = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
assert (REPO_DIR / "hooklink/Resources/tools/hl_resolution.py").is_file()
assert (REPO_DIR / ".github/workflows/hooklink2ios.yml").is_file()
print("Checkout:", REPO_HEAD)
print("Branch:", subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip())
print("bsdtar:", subprocess.check_output(["bsdtar", "--version"], text=True).splitlines()[0])
print("Workspace pré-existente:", Path("/content/hooklink2ios-work").exists())

## 1. Carregar e conferir o código completo

O helper abaixo é incorporado neste notebook para que o conteúdo operacional fique visível aqui, em vez de depender de uma célula curta que apenas chama código externo. O SHA-256 do texto incorporado é comparado com o helper versionado na branch antes da execução.

In [ ]:
HELPER_SOURCE = r'''#!/usr/bin/env python3
"""Linux/Colab intake + RomFS preparation for HookLink2iOS."""
from __future__ import annotations
import base64, email.message, hashlib, json, os, re, shutil, subprocess, sys, time, uuid, zipfile
from pathlib import Path, PurePosixPath
from urllib.parse import parse_qs, unquote, urlsplit
import requests
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError

REPO = "joao9vegeto-stack/KintTany"
BRANCH = "HookLink2iOS"
WORKFLOW_NAME = "HookLink2iOS Ocarina native build"
HANDOFF = "hooklink2ios/handoff.json"
WORKFLOW_NAME = "HookLink2iOS Ocarina native build"
ENGINE = "https://github.com/dsi-hub/nxnative.git"
ENGINE_PIN = "cf2a2fecc6ce3fb841131ffeffa90915cdadf109"
TITLE_ID = "0400486024DEC000"
VERSION = 131072
GPU = "g18p"  # iPhone 17 / 17 Pro A19 family; change only after validating the target GPU.
CHUNK = 64 * 1024 * 1024
FOLDER = "application/vnd.google-apps.folder"

def drive_execute(request, label="Google Drive API", retries=12):
    """Retry transient Drive API failures (429/5xx) with exponential backoff."""
    for attempt in range(retries + 1):
        try:
            return request.execute(num_retries=0)
        except HttpError as exc:
            status=int(getattr(exc.resp, "status", 0) or 0)
            if status not in (408, 429, 500, 502, 503, 504) or attempt >= retries:
                raise
            delay=min(90, 2 ** min(attempt + 1, 6))
            print(f"[HookLink2iOS] {label}: HTTP {status}; nova tentativa "
                  f"{attempt + 1}/{retries} em {delay}s.", flush=True)
            time.sleep(delay)
        except (BrokenPipeError, ConnectionError, TimeoutError, OSError) as exc:
            if attempt >= retries:
                raise
            delay=min(90, 2 ** min(attempt + 1, 6))
            print(f"[HookLink2iOS] {label}: conexão instável; nova tentativa "
                  f"{attempt + 1}/{retries} em {delay}s ({type(exc).__name__}).", flush=True)
            time.sleep(delay)

def run(cmd, cwd=None):
    print("[HookLink2iOS] $", " ".join(map(str, cmd)), flush=True)
    p = subprocess.run(list(map(str,cmd)), cwd=cwd, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, text=True, errors="replace")
    print(p.stdout, end="", flush=True)
    if p.returncode:
        raise RuntimeError(f"Comando falhou ({p.returncode}): {cmd[0]}")
    return p.stdout

def file_hashes(path):
    path=Path(path)
    sha, md5, size = hashlib.sha256(), hashlib.md5(), 0
    total=path.stat().st_size
    last_report=0
    print(f"[HookLink2iOS] Calculando SHA-256/MD5: {path.name} ({total:,} bytes)",flush=True)
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(16*1024*1024),b""):
            sha.update(block); md5.update(block); size += len(block)
            if size-last_report >= 1024*1024*1024 or size==total:
                print(f"[HookLink2iOS] Hash {path.name}: {size/total*100:.1f}% ({size:,}/{total:,} bytes)",flush=True)
                last_report=size
    return {"size":size,"sha256":sha.hexdigest(),"md5":md5.hexdigest()}

def safe_name(name):
    name = Path(unquote(str(name))).name.strip()
    if not name or name in (".",".."): raise ValueError("Nome de arquivo inválido.")
    return re.sub(r"[^A-Za-z0-9._ -]","_",name)

def source_key(url):
    parsed=urlsplit(url.strip())
    if parsed.scheme.lower()!="https" or not parsed.netloc or not parsed.path:
        raise ValueError("Forneça uma URL HTTPS completa.")
    stable=f"https://{parsed.netloc.lower()}{parsed.path}"
    return hashlib.sha256(stable.encode()).hexdigest()

def download_url(url, dst):
    if not url.startswith("https://"): raise ValueError("A origem precisa usar HTTPS.")
    dst.parent.mkdir(parents=True,exist_ok=True)
    part=dst.with_name(dst.name+".part"); total=last=0
    with requests.get(url,stream=True,allow_redirects=True,timeout=(30,180),
        headers={"User-Agent":"HookLink2iOS-Colab/2.0","Accept-Encoding":"identity"}) as r:
        r.raise_for_status()
        disposition=r.headers.get("Content-Disposition","")
        if not disposition:
            disposition=parse_qs(urlsplit(url).query).get("response-content-disposition",[""])[0]
        msg=email.message.Message()
        if disposition: msg["content-disposition"]=disposition
        filename=msg.get_filename() or Path(unquote(urlsplit(url).path)).name
        filename=safe_name(filename)
        target=dst.parent/filename
        expected=int(r.headers.get("content-length") or 0)
        with part.open("wb") as out:
            for block in r.iter_content(8*1024*1024):
                if not block: continue
                out.write(block); total+=len(block)
                if total-last>=512*1024*1024:
                    print(f"[HookLink2iOS] Download {total:,}/{expected or '?'} bytes",flush=True); last=total
        if total<4 or (expected and total!=expected):
            raise ValueError(f"Download incompleto: {total:,}/{expected or '?'} bytes.")
    part.replace(target)
    h=file_hashes(target)
    print(f"[HookLink2iOS] Download concluído: {filename} | {h['size']:,} bytes | SHA-256 {h['sha256']}")
    return target,filename,h

def download_drive_source(drive, file_id, dst):
    meta = drive.files().get(fileId=file_id,fields="id,name,size,mimeType",supportsAllDrives=True).execute()
    if meta.get("mimeType")==FOLDER: raise ValueError("O ID aponta para uma pasta, não um arquivo.")
    part=dst.with_name(dst.name+".part"); dst.parent.mkdir(parents=True,exist_ok=True)
    req=drive.files().get_media(fileId=file_id,supportsAllDrives=True)
    with part.open("wb") as out:
        dl=MediaIoBaseDownload(out,req,chunksize=CHUNK); done=False
        while not done:
            status,done=dl.next_chunk(num_retries=8)
            if status: print(f"[HookLink2iOS] Drive download {status.progress()*100:.1f}%",flush=True)
    if int(meta.get("size") or 0) and part.stat().st_size != int(meta["size"]):
        part.unlink(missing_ok=True); raise ValueError("Tamanho do arquivo no Drive não confere.")
    part.replace(dst)
    return meta["name"]

def extract_archive(archive,dest):
    if not shutil.which("bsdtar"): raise RuntimeError("bsdtar ausente; instale libarchive-tools no Colab.")
    listing=run(["bsdtar","-tf",archive])
    entries=[x.strip() for x in listing.splitlines() if x.strip()]
    if not entries: raise ValueError("Arquivo compactado vazio.")
    for raw in entries:
        norm=raw.replace("\\","/")
        member=PurePosixPath(norm)
        if member.is_absolute() or re.match(r"^[A-Za-z]:",norm) or ".." in member.parts:
            raise ValueError(f"Caminho inseguro no arquivo: {raw}")
    verbose=run(["bsdtar","-tvf",archive])
    for line in verbose.splitlines():
        mode=line.split(maxsplit=1)[0] if line.split() else ""
        if mode.startswith(("l","h")): raise ValueError("Arquivo contém link simbólico/duro; extração recusada.")
    shutil.rmtree(dest,ignore_errors=True); dest.mkdir(parents=True)
    run(["bsdtar","-xf",archive,"-C",dest])
    nups=sorted(p for p in dest.rglob("*") if p.is_file() and p.suffix.lower()==".nup")
    if len(nups)!=1:
        raise ValueError(f"Esperado exatamente um NUP no arquivo; encontrados {len(nups)}.")
    return nups[0]

def clone_engine(dest):
    shutil.rmtree(dest,ignore_errors=True); dest.parent.mkdir(parents=True,exist_ok=True)
    run(["git","init",dest]); run(["git","-C",dest,"remote","add","origin",ENGINE])
    run(["git","-C",dest,"fetch","--depth","1","origin",ENGINE_PIN])
    run(["git","-C",dest,"checkout","--detach","FETCH_HEAD"])
    sha=subprocess.check_output(["git","-C",dest,"rev-parse","HEAD"],text=True).strip()
    if sha!=ENGINE_PIN: raise RuntimeError(f"nxnative pin divergente: {sha}")

def ensure_folder(drive,parent,name):
    qname=name.replace("'","\\'")
    res=drive.files().list(q=f"'{parent}' in parents and name = '{qname}' and mimeType = '{FOLDER}' and trashed = false",
        spaces="drive",pageSize=100,fields="files(id,name,webViewLink)",supportsAllDrives=True,
        includeItemsFromAllDrives=True).execute(num_retries=5)
    if res.get("files"): return res["files"][0]
    return drive.files().create(body={"name":name,"mimeType":FOLDER,"parents":[parent]},
        fields="id,name,webViewLink",supportsAllDrives=True).execute(num_retries=5)

def upload(drive,path,parent,name,app_properties=None,description=None,precomputed_hashes=None):
    h=precomputed_hashes or file_hashes(path)
    body={"name":name,"parents":[parent]}
    if app_properties: body["appProperties"]={str(k):str(v) for k,v in app_properties.items()}
    if description: body["description"]=description[:25000]
    media=MediaFileUpload(str(path),mimetype="application/octet-stream",chunksize=CHUNK,resumable=True)
    req=drive.files().create(body=body,media_body=media,
        fields="id,name,size,md5Checksum,webViewLink,description,appProperties",supportsAllDrives=True)
    item=None
    failures=0
    while item is None:
        try:
            status,item=req.next_chunk(num_retries=8)
            failures=0
            if status: print(f"[HookLink2iOS] Drive upload {name}: {status.progress()*100:.1f}%",flush=True)
        except (BrokenPipeError, ConnectionError, TimeoutError, OSError) as exc:
            failures += 1
            if failures > 12:
                raise RuntimeError(
                    f"Upload interrompido após {failures} tentativas; o arquivo local foi preservado: {path}"
                ) from exc
            delay=min(60, 2 ** min(failures, 6))
            print(f"[HookLink2iOS] Conexão Drive caiu durante {name}; retomando a sessão "
                  f"({failures}/12) em {delay}s...", flush=True)
            time.sleep(delay)
    if int(item.get("size",-1))!=h["size"] or str(item.get("md5Checksum","")).lower()!=h["md5"]:
        raise RuntimeError(f"Upload Drive falhou na verificação: {name}")
    return {"id":item["id"],"name":name,**h,"web_view_url":item.get("webViewLink")}

def list_children(drive,folder_id):
    out=[]; token=None
    while True:
        res=drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
            spaces="drive",pageSize=100,pageToken=token,
            fields="nextPageToken,files(id,name,size,md5Checksum,appProperties,webViewLink)",
            supportsAllDrives=True,includeItemsFromAllDrives=True).execute(num_retries=5)
        out.extend(res.get("files",[])); token=res.get("nextPageToken")
        if not token: return out

def find_cached_nup(drive,folder_id,key):
    matches=[x for x in list_children(drive,folder_id)
        if (x.get("appProperties") or {}).get("hooklink2ios_source_key")==key
        and x.get("name","").lower().endswith(".nup")]
    return matches[0] if matches else None

def download_verified_nup(drive,record,destination):
    meta=drive.files().get(fileId=record["id"],fields="id,name,size,md5Checksum",supportsAllDrives=True).execute()
    part=destination.with_name(destination.name+".part")
    req=drive.files().get_media(fileId=record["id"],supportsAllDrives=True)
    with part.open("wb") as out:
        dl=MediaIoBaseDownload(out,req,chunksize=CHUNK); done=False
        while not done:
            status,done=dl.next_chunk(num_retries=8)
            if status: print(f"[HookLink2iOS] Drive → Colab {record['name']}: {status.progress()*100:.1f}%",flush=True)
    h=file_hashes(part)
    expected=(record.get("appProperties") or {}).get("hooklink2ios_nup_sha256","")
    if int(meta.get("size",-1))!=h["size"] or (expected and h["sha256"]!=expected):
        part.unlink(missing_ok=True); raise RuntimeError("NUP no Drive falhou na verificação SHA/size.")
    part.replace(destination)
    return {"id":record["id"],"name":record["name"],**h,"web_view_url":record.get("webViewLink")}

def publish(token,handoff):
    if not token: raise ValueError("GitHub PAT obrigatório.")
    url=f"https://api.github.com/repos/{REPO}/contents/{HANDOFF}"
    headers={"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json",
             "X-GitHub-Api-Version":"2022-11-28","User-Agent":"HookLink2iOS-Colab"}
    old=requests.get(url,params={"ref":BRANCH},headers=headers,timeout=60)
    body={"message":f"HookLink2iOS handoff {handoff['request_id']}",
        "content":base64.b64encode(json.dumps(handoff,indent=2).encode()).decode(),"branch":BRANCH}
    if old.status_code==200: body["sha"]=old.json()["sha"]
    elif old.status_code!=404: raise RuntimeError(f"Não foi possível ler handoff atual (HTTP {old.status_code}).")
    r=requests.put(url,headers=headers,json=body,timeout=60)
    if r.status_code not in (200,201): raise RuntimeError(f"GitHub recusou handoff (HTTP {r.status_code}): {r.text[:500]}")
    c=r.json().get("commit",{})
    return {"commit_sha":c.get("sha"),"commit_url":c.get("html_url"),
            "actions_url":f"https://github.com/{REPO}/actions"}

def prepare(*,source_url,drive_parent_folder_id,drive,github_token,repo_root,
            work_root=Path("/content/hooklink2ios-work")):
    if sys.version_info<(3,12): raise RuntimeError("Python 3.12+ obrigatório.")
    key=source_key(source_url); request_id=str(uuid.uuid4())
    inp=work_root/"input"; extracted=work_root/"unpacked"
    engine=work_root/"nxnative"; build=work_root/"linux-build"
    inp.mkdir(parents=True,exist_ok=True); work_root.mkdir(parents=True,exist_ok=True)
    parent=drive_execute(drive.files().get(fileId=drive_parent_folder_id,fields="id,mimeType",supportsAllDrives=True), "consulta da pasta raiz")
    if parent.get("mimeType")!=FOLDER: raise ValueError("DRIVE_FOLDER_ID não é uma pasta.")
    project=ensure_folder(drive,drive_parent_folder_id,"HookLink2iOS")
    inputs=ensure_folder(drive,project["id"],"Inputs")
    cached=find_cached_nup(drive,inputs["id"],key)
    nup=inp/"game.nup"; archive=None; archive_extracted=False; source_filename=""
    # Recuperação local: se a sessão anterior caiu durante o upload, reutiliza o NUP
    # já extraído em /content e não baixa nem extrai novamente o arquivo de origem.
    local_sources=[p for p in inp.iterdir() if p.is_file() and p.name not in
        ("game.nup","game.nup.part") and not p.name.endswith(".part")]
    if nup.is_file() and nup.stat().st_size>0:
        source_filename=local_sources[0].name if local_sources else "recuperado-local.nup"
        archive_extracted = (
            Path(source_filename).suffix.lower() in
            (".rar", ".zip", ".7z", ".tar", ".tgz", ".gz", ".xz", ".bz2")
        )
        print(f"[HookLink2iOS] NUP LOCAL RECUPERADO: {nup.stat().st_size:,} bytes; sem download/extração.")
        nh=file_hashes(nup)
        cache_name=f"{Path(source_filename).stem[:55]}_{key[:10]}.nup"
        nup_item=upload(drive,nup,inputs["id"],cache_name,
            app_properties={"hooklink2ios_source_key":key,"hooklink2ios_nup_sha256":nh["sha256"],
                "hooklink2ios_source_filename":source_filename[:120],
                "hooklink2ios_archive_extracted":str(archive_extracted).lower()},
            description=f"HookLink2iOS input cache | source={source_filename} | key={key} | sha256={nh['sha256']}",
            precomputed_hashes=nh)
        for candidate in local_sources:
            candidate.unlink(missing_ok=True)
        print("[HookLink2iOS] NUP local persistido e verificado no Drive.")
    elif cached:
        print("[HookLink2iOS] CACHE HIT: NUP já está no Drive; ignorando o download externo.")
        nup_item=download_verified_nup(drive,cached,nup)
        source_filename=(cached.get("appProperties") or {}).get("hooklink2ios_source_filename",cached["name"])
        archive_extracted=(cached.get("appProperties") or {}).get("hooklink2ios_archive_extracted")=="true"
    else:
        source,source_filename,_=download_url(source_url,inp/"source")
        ext=Path(source_filename).suffix.lower()
        is_archive=ext in (".rar",".zip",".7z",".tar",".tgz",".gz",".xz",".bz2") or source_filename.lower().endswith((".tar.gz",".tar.xz",".tar.zst",".tar.bz2"))
        if ext==".nup":
            if source.resolve()!=nup.resolve(): shutil.copy2(source,nup)
        elif is_archive:
            extracted_nup=extract_archive(source,extracted)
            shutil.copy2(extracted_nup,nup); archive=source; archive_extracted=True
        else:
            raise ValueError(f"Formato {ext} não suportado; esperado .nup/.rar/.zip/.7z ou tar compactado.")
        nh=file_hashes(nup)
        cache_name=f"{Path(source_filename).stem[:55]}_{key[:10]}.nup"
        nup_item=upload(drive,nup,inputs["id"],cache_name,
            app_properties={"hooklink2ios_source_key":key,"hooklink2ios_nup_sha256":nh["sha256"],
                "hooklink2ios_source_filename":source_filename[:120],
                "hooklink2ios_archive_extracted":str(archive_extracted).lower()},
            description=f"HookLink2iOS input cache | source={source_filename} | key={key} | sha256={nh['sha256']}")
        # Keep the validated NUP on Drive; discard the source archive only after upload verification.
        if source.exists() and source.resolve()!=nup.resolve(): source.unlink()
        shutil.rmtree(extracted,ignore_errors=True)
        print("[HookLink2iOS] NUP persistido e verificado em HookLink2iOS/Inputs.")
    if not nup.is_file() or not nup.stat().st_size: raise RuntimeError("NUP ausente após download/cache.")
    clone_engine(engine)
    out=run([sys.executable,engine/"scripts/pack_nup.py",nup,"--version",str(VERSION),"--work-dir",build],cwd=engine)
    if f"{TITLE_ID} version {VERSION}" not in out: raise ValueError("Title ID/update não corresponde ao alvo; handoff cancelado.")
    romfs=build/"extract"/f"v{VERSION}"/"RomFS.img"
    if not romfs.is_file() or not romfs.stat().st_size: raise RuntimeError("RomFS.img não foi criado.")
    identity=run([sys.executable,engine/"dm/tools/romfs_image.py","--identity",romfs,TITLE_ID,f"v{VERSION}",ENGINE_PIN]).strip().splitlines()[-1]
    overlay=work_root/"res_overlay"/"Pack"/"Bootup.Nin_Ounce_NVN2.pack.zs"
    overlay.parent.mkdir(parents=True,exist_ok=True)
    run([sys.executable,repo_root/"hooklink/Resources/tools/hl_resolution.py",engine,romfs,overlay,"0.75","95"])
    if not overlay.is_file() or not overlay.stat().st_size: raise RuntimeError("Overlay não foi gerado.")
    runfolder=drive.files().create(body={"name":f"HookLink2iOS-{request_id}","mimeType":FOLDER,"parents":[project["id"]]},
        fields="id,name,webViewLink",supportsAllDrives=True).execute()
    romfs_item=upload(drive,romfs,runfolder["id"],"romfs.img")
    overlay_item=upload(drive,overlay,runfolder["id"],"Bootup.Nin_Ounce_NVN2.pack.zs")
    handoff={"schema":2,"project":"HookLink2iOS","branch":BRANCH,"request_id":request_id,
        "title_id":TITLE_ID,"package_version":f"v{VERSION}","package_version_number":VERSION,
        "gpu_arch":GPU,"engine":{"repository":ENGINE,"branch":"hooklink","commit":ENGINE_PIN},
        "source":{"filename":source_filename,"url_fingerprint":key,"archive_extracted":archive_extracted,
                  "archive_deleted_after_extraction":archive_extracted},
        "drive":{"parent_folder_id":drive_parent_folder_id,"project_folder_id":project["id"],
            "inputs_folder_id":inputs["id"],"run_folder_id":runfolder["id"],"run_folder_url":runfolder.get("webViewLink")},
        "nup":nup_item,"romfs":{**romfs_item,"identity":identity},"resolution_overlay":overlay_item,
        "builder":{"platform":"ios","romfs_mode":"separate","runner":"macos-26","xcode":"26.6.0"},
        "created_at_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime())}
    pushed=publish(github_token,handoff)
    print("[HookLink2iOS] Handoff publicado somente em HookLink2iOS; commit:",pushed["commit_sha"])
    return {"handoff":handoff,"github":pushed}

def github_headers(token):
    if not token: raise ValueError("GitHub PAT obrigatório.")
    return {"Authorization":f"Bearer {token}","Accept":"application/vnd.github+json",
        "X-GitHub-Api-Version":"2022-11-28","User-Agent":"HookLink2iOS-Colab"}

def github_get(session,url,params=None):
    r=session.get(url,params=params,timeout=(30,120))
    if not r.ok: raise RuntimeError(f"GitHub API HTTP {r.status_code}: {r.text[:800]}")
    return r.json()

def recover_current_handoff(*,github_token,source_url):
    key=source_key(source_url); session=requests.Session(); session.headers.update(github_headers(github_token))
    url=f"https://api.github.com/repos/{REPO}/contents/{HANDOFF}"
    r=session.get(url,params={"ref":BRANCH},timeout=(30,90))
    if r.status_code==404: return None
    if not r.ok: raise RuntimeError(f"Leitura do handoff falhou: HTTP {r.status_code}: {r.text[:800]}")
    h=json.loads(base64.b64decode(r.json()["content"]).decode("utf-8"))
    if h.get("project")!="HookLink2iOS" or h.get("branch")!=BRANCH or h.get("source",{}).get("url_fingerprint")!=key: return None
    commits=github_get(session,f"https://api.github.com/repos/{REPO}/commits",
        {"sha":BRANCH,"path":HANDOFF,"per_page":1})
    if not commits: raise RuntimeError("Commit do handoff não foi encontrado.")
    return {"handoff":h,"commit_sha":commits[0]["sha"],
        "commit_url":f"https://github.com/{REPO}/commit/{commits[0]['sha']}",
        "actions_url":f"https://github.com/{REPO}/actions"}

def verify_handoff_drive_assets(drive,h):
    for key in ("nup","romfs","resolution_overlay"):
        rec=h[key]
        remote=drive.files().get(fileId=rec["id"],fields="id,size,md5Checksum",supportsAllDrives=True).execute()
        if int(remote.get("size",-1))!=int(rec["size"]): raise RuntimeError(f"{key} size diverge no Drive.")
        if rec.get("md5") and remote.get("md5Checksum","").lower()!=rec["md5"].lower(): raise RuntimeError(f"{key} MD5 diverge no Drive.")

def wait_for_actions_artifact(*,github_token,expected_commit,timeout_seconds=21600,poll_seconds=30):
    session=requests.Session(); session.headers.update(github_headers(github_token))
    url=f"https://api.github.com/repos/{REPO}/actions/runs"; deadline=time.time()+timeout_seconds; run_data=None
    while time.time()<deadline:
        payload=github_get(session,url,{"branch":BRANCH,"event":"push","per_page":100,"page":1})
        matches=[x for x in payload.get("workflow_runs",[]) if x.get("head_sha")==expected_commit
            and x.get("head_branch")==BRANCH and (
                str(x.get("path","")).endswith("hooklink2ios.yml")
                or WORKFLOW_NAME.lower() in str(x.get("name","")).lower()
            )]
        if matches:
            run_data=sorted(matches,key=lambda x:x.get("id",0),reverse=True)[0]
            print("[HookLink2iOS] Actions run",run_data["run_number"],run_data.get("status"),run_data.get("html_url"),flush=True)
            if run_data.get("status")=="completed": break
        else: print("[HookLink2iOS] Aguardando workflow do commit",expected_commit,flush=True)
        time.sleep(poll_seconds)
    if not run_data or run_data.get("status")!="completed": raise TimeoutError("Workflow não concluiu no limite de espera.")
    if run_data.get("conclusion")!="success": raise RuntimeError(f"Workflow {run_data.get('conclusion')}: {run_data.get('html_url')}")
    artifacts=github_get(session,run_data["artifacts_url"],{"per_page":100,"page":1}).get("artifacts",[])
    name=f"HookLink2iOS-Ocarina-{run_data['run_number']}"
    found=[x for x in artifacts if x.get("name")==name and not x.get("expired")]
    if len(found)!=1: raise RuntimeError(f"Artifact {name} ausente/expirado.")
    return {"run":run_data,"artifact":found[0],"headers":dict(session.headers)}

def transfer_actions_artifact_to_drive(*,github_token,expected_commit,drive,run_folder_id):
    info=wait_for_actions_artifact(github_token=github_token,expected_commit=expected_commit)
    artifact=info["artifact"]; run_data=info["run"]
    url=f"https://api.github.com/repos/{REPO}/actions/artifacts/{artifact['id']}/zip"
    work=Path("/content/hooklink2ios-artifact-transfer"); work.mkdir(parents=True,exist_ok=True)
    zip_path=work/"artifact.zip"; part=zip_path.with_suffix(".part"); total=0
    r=requests.get(url,headers=info["headers"],stream=True,allow_redirects=True,timeout=(30,180)); r.raise_for_status()
    with r:
        with part.open("wb") as out:
            for block in r.iter_content(8*1024*1024):
                if block: out.write(block); total+=len(block)
    expected_size=int(artifact.get("size_in_bytes") or 0)
    if expected_size and total!=expected_size: part.unlink(missing_ok=True); raise RuntimeError("ZIP do artifact incompleto.")
    zip_sha=file_hashes(part)["sha256"]; digest=str(artifact.get("digest") or "").split(":",1)[-1].lower()
    if digest and zip_sha.lower()!=digest: part.unlink(missing_ok=True); raise RuntimeError("Digest do ZIP diverge do GitHub.")
    part.replace(zip_path)
    dest=work/"extracted"; shutil.rmtree(dest,ignore_errors=True); dest.mkdir()
    with zipfile.ZipFile(zip_path) as z:
        members=z.infolist()
        for m in members:
            path=PurePosixPath(m.filename)
            if path.is_absolute() or ".." in path.parts: raise ValueError("Caminho inseguro no artifact.")
        ipa=[m for m in members if m.filename.lower().endswith(".ipa")]
        report=[m for m in members if PurePosixPath(m.filename).name=="build-report.json"]
        overlays=[m for m in members if PurePosixPath(m.filename).name=="Bootup.Nin_Ounce_NVN2.pack.zs"]
        if len(ipa)!=1 or len(report)!=1 or len(overlays)!=1: raise RuntimeError("Artifact não contém IPA/report/overlay únicos.")
        paths={}
        for m in (ipa[0],report[0],overlays[0]):
            p=dest/PurePosixPath(m.filename).name
            with z.open(m) as src,p.open("wb") as out: shutil.copyfileobj(src,out,8*1024*1024)
            paths[p.name]=p
    report_data=json.loads(paths["build-report.json"].read_text())
    ih=file_hashes(paths[ipa[0].filename.split("/")[-1]])
    if report_data.get("project")!="HookLink2iOS" or report_data.get("branch")!=BRANCH: raise RuntimeError("Proveniência do build-report incorreta.")
    if int(report_data.get("ipa",{}).get("size",-1))!=ih["size"] or report_data.get("ipa",{}).get("sha256","").lower()!=ih["sha256"]: raise RuntimeError("Hash/tamanho da IPA diverge do build-report.")
    if report_data.get("commit")!=expected_commit:
        raise RuntimeError("Commit no build-report não corresponde ao commit do workflow.")
    build_folder=ensure_folder(drive,run_folder_id,
        f"actions-{run_data['run_number']}-{run_data['id']}")
    provenance=f"HookLink2iOS | branch={BRANCH} | commit={expected_commit} | run_id={run_data['id']} | artifact={artifact['name']} | run_url={run_data.get('html_url')}"
    ipa_item=upload(drive,paths[ipa[0].filename.split("/")[-1]],build_folder["id"],"OcarinaOfTime-HookLink2iOS.ipa",description=provenance)
    upload(drive,paths[overlays[0].filename.split("/")[-1]],build_folder["id"],"Bootup.Nin_Ounce_NVN2.pack.zs",description=provenance)
    upload(drive,paths["build-report.json"],build_folder["id"],"build-report.json",description=provenance)
    remote=drive.files().get(fileId=ipa_item["id"],fields="id,name,size,md5Checksum,webViewLink,description",supportsAllDrives=True).execute()
    if int(remote.get("size",-1))!=ih["size"] or remote.get("md5Checksum","").lower()!=ih["md5"] or remote.get("description")!=provenance: raise RuntimeError("IPA no Drive falhou na verificação final.")
    zip_path.unlink(missing_ok=True)
    return {"run":run_data,"artifact":artifact,"ipa":remote,"ipa_sha256":ih["sha256"],
        "drive_folder":build_folder,"commit_sha":expected_commit,"run_url":run_data.get("html_url"),
        "commit_url":f"https://github.com/{REPO}/commit/{expected_commit}"}

if __name__=="__main__":
    print("Importe prepare() pelo notebook Colab e cole uma URL HTTPS de NUP/arquivo compactado.")

'''
HELPER_SHA256 = hashlib.sha256(HELPER_SOURCE.encode("utf-8")).hexdigest()
REPO_HELPER = REPO_DIR / "colab/hooklink2ios_prepare.py"
REPO_HELPER_SHA256 = hashlib.sha256(REPO_HELPER.read_bytes()).hexdigest()
assert HELPER_SHA256 == REPO_HELPER_SHA256, (
    "O helper embutido no notebook diverge do helper commitado na branch. "
    f"Notebook={HELPER_SHA256}; branch={REPO_HELPER_SHA256}"
)
exec(compile(HELPER_SOURCE, str(REPO_HELPER), "exec"), globals())
print("Helper conferido:", HELPER_SHA256)
print("Funções: prepare | recover_current_handoff | verify_handoff_drive_assets | transfer_actions_artifact_to_drive")

## 2. Autenticar o Drive já configurado

Usa a autenticação OAuth da conta Google e a pasta raiz do notebook Link2iOS. Para o Actions, reutiliza o secret permanente `GOOGLE_SERVICE_ACCOUNT_JSON` que já existe no repositório e é consumido pelo workflow Link2iOS. Não pede novo ID, JSON, service account ou permissão Secrets no PAT.

In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)
root_folder = drive.files().get(fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType,webViewLink").execute()
assert root_folder.get("mimeType") == "application/vnd.google-apps.folder", root_folder
print("Google Drive autenticado:", root_folder.get("name"))
print("Pasta raiz reutilizada:", DRIVE_FOLDER_ID)
print("Cache NUP: HookLink2iOS/Inputs")

## 3. Colar diretamente a URL do arquivo

**Cole a URL inteira no primeiro prompt.** Não selecione modo e não informe nome de arquivo. A entrada fica oculta para preservar os parâmetros de assinatura do link.

In [ ]:
from getpass import getpass

SOURCE_URL = getpass("Cole a URL HTTPS completa do .nup/.rar/.zip/.7z: ").strip()
assert SOURCE_URL.startswith("https://"), "Cole uma URL HTTPS completa."
SOURCE_KEY = source_key(SOURCE_URL)

GITHUB_PAT = getpass("GitHub Fine-grained PAT (Contents: Read/Write; Actions: Read): ").strip()
assert GITHUB_PAT, "PAT obrigatório para publicar o handoff e ler o artifact do Actions."

# Confere acesso antes de começar um download potencialmente grande.
preflight = requests.get(f"https://api.github.com/repos/{REPO}",
    headers=github_headers(GITHUB_PAT), timeout=(30, 90))
preflight.raise_for_status()
repo_permissions = preflight.json().get("permissions", {})
assert repo_permissions.get("push") is True, "O PAT não tem permissão Contents: write neste repositório."
branch_check = requests.get(f"https://api.github.com/repos/{REPO}/branches/{BRANCH}",
    headers=github_headers(GITHUB_PAT), timeout=(30, 90))
branch_check.raise_for_status()
print("URL recebida em memória; não será exibida nem publicada.")
print("Fingerprint estável:", SOURCE_KEY)
print("Repositório e branch autorizados:", REPO, BRANCH)

## 4. Recuperar a preparação ou executar a primeira preparação

Se já existir handoff íntegro para a mesma URL na branch, o notebook valida NUP/RomFS/overlay no Drive, reutiliza esses arquivos e publica um novo identificador de tentativa para iniciar apenas outro build iOS. Não repete download, extração nem geração de RomFS. Se ainda não houver handoff recuperável, usa o cache do NUP em `HookLink2iOS/Inputs` quando disponível; caso contrário, baixa a URL, extrai exatamente um NUP do arquivo compactado, persiste e verifica esse NUP no Drive antes de apagar o arquivo compactado local. A preparação Linux continua a partir do NUP persistido.


In [ ]:
RECOVERED = recover_current_handoff(github_token=GITHUB_PAT, source_url=SOURCE_URL)

if RECOVERED is not None:
    HANDOFF = RECOVERED["handoff"]
    verify_handoff_drive_assets(drive, HANDOFF)
    # Assets de Linux reutilizados: cria apenas um novo disparo do build nativo iOS.
    import uuid
    HANDOFF["build_attempt"] = str(uuid.uuid4())
    GITHUB_RESULT = publish(GITHUB_PAT, HANDOFF)
    HANDOFF_COMMIT_SHA = GITHUB_RESULT["commit_sha"]
    RESULT = {"handoff": HANDOFF, "github": GITHUB_RESULT}
    print("CACHE/HANDOFF RECUPERADO: sem novo download, extração ou geração de RomFS; apenas novo build iOS.")
else:
    RESULT = prepare(
        source_url=SOURCE_URL,
        drive_parent_folder_id=DRIVE_FOLDER_ID,
        drive=drive,
        github_token=GITHUB_PAT,
        repo_root=REPO_DIR,
        work_root=Path("/content/hooklink2ios-work"),
    )
    HANDOFF = RESULT["handoff"]
    HANDOFF_COMMIT_SHA = RESULT["github"]["commit_sha"]
    print("Primeira preparação concluída; NUP persistido no Drive.")

assert HANDOFF["title_id"] == "0400486024DEC000"
assert HANDOFF["package_version"] == "v131072"
assert HANDOFF["branch"] == "HookLink2iOS"
assert HANDOFF_COMMIT_SHA
print("Commit que identifica o build:", HANDOFF_COMMIT_SHA)
print("Commit URL:", f"https://github.com/{REPO}/commit/{HANDOFF_COMMIT_SHA}")
print("Actions:", RESULT["github"]["actions_url"])

## 5. Auditar o handoff

Confere os identificadores e hashes que serão consumidos pelo runner macOS-26. O NUP é referenciado na pasta persistente `Inputs`; RomFS e overlay estão na pasta desta execução.

In [ ]:
print("=" * 72)
print("HOOKLINK2iOS — HANDOFF VALIDADO")
print("=" * 72)
print("Request:", HANDOFF["request_id"])
print("Title ID:", HANDOFF["title_id"], "| Pacote:", HANDOFF["package_version"])
print("Origem:", HANDOFF["source"]["filename"])
print("Arquivo compactado extraído:", HANDOFF["source"]["archive_extracted"])
print("NUP:", f'{HANDOFF["nup"]["size"]:,}', "bytes | SHA-256:", HANDOFF["nup"]["sha256"])
print("RomFS:", f'{HANDOFF["romfs"]["size"]:,}', "bytes | SHA-256:", HANDOFF["romfs"]["sha256"])
print("Overlay:", f'{HANDOFF["resolution_overlay"]["size"]:,}', "bytes | SHA-256:", HANDOFF["resolution_overlay"]["sha256"])
print("nxnative:", HANDOFF["engine"]["commit"])
print("Pasta desta execução:", HANDOFF["drive"].get("run_folder_url") or HANDOFF["drive"]["run_folder_id"])
print("Commit Actions:", HANDOFF_COMMIT_SHA)
print("=" * 72)

## 6. Esperar o Actions e transferir IPA → Drive

Acompanha somente o workflow iniciado pelo commit exato do handoff. Não seleciona execução de outro jogo. O ZIP é conferido contra o digest SHA-256 da API do GitHub; a IPA é verificada contra o relatório do build, incluindo o commit. Depois, IPA, overlay e relatório são enviados com o OAuth do Colab para uma subpasta exclusiva da execução do Actions, sem sobrescrever resultados anteriores.


In [ ]:
TRANSFER = transfer_actions_artifact_to_drive(
    github_token=GITHUB_PAT,
    expected_commit=HANDOFF_COMMIT_SHA,
    drive=drive,
    run_folder_id=HANDOFF["drive"]["run_folder_id"],
)
assert TRANSFER["commit_sha"] == HANDOFF_COMMIT_SHA
assert TRANSFER["ipa"].get("webViewLink")
print("IPA enviada e verificada no Google Drive.")


## 7. Resumo final e proveniência

In [ ]:
RUN = TRANSFER["run"]
ARTIFACT = TRANSFER["artifact"]
IPA = TRANSFER["ipa"]
print("=" * 72)
print("HOOKLINK2iOS — IPA PRONTA NO DRIVE")
print("=" * 72)
print("Branch:", BRANCH)
print("Commit SHA:", TRANSFER["commit_sha"])
print("Commit:", TRANSFER["commit_url"])
print("Workflow run:", RUN["run_number"], "| ID:", RUN["id"])
print("Execução:", TRANSFER["run_url"])
print("Artifact:", ARTIFACT["name"], "| ID:", ARTIFACT["id"])
print("IPA:", IPA["name"])
print("Tamanho:", f'{int(IPA["size"]):,}', "bytes")
print("SHA-256:", TRANSFER["ipa_sha256"])
print("MD5 Drive:", IPA["md5Checksum"])
print("Link IPA Drive:", IPA["webViewLink"])
print("Pasta exclusiva deste build:", TRANSFER["drive_folder"].get("webViewLink") or
      "https://drive.google.com/drive/folders/" + TRANSFER["drive_folder"]["id"])
print("Pasta:", HANDOFF["drive"].get("run_folder_url") or
      "https://drive.google.com/drive/folders/" + HANDOFF["drive"]["run_folder_id"])
print("Overlay e build-report.json também foram enviados.")
print("=" * 72)

## Retomada e erros esperados

Se o Colab desconectar depois do handoff, abra novamente este notebook, execute as células em ordem e cole a mesma URL e PAT. O handoff recupera os arquivos já persistidos; o NUP/RomFS/overlay não serão baixados/preparados novamente. Um novo build iOS será disparado e seus resultados irão para uma subpasta própria.

- **HTTP 403/404 na URL:** link assinado expirado/revogado; gere outro e cole novamente. Se o NUP já estiver no Drive, a recuperação não precisa usar a URL.
- **Nenhum NUP ou mais de um NUP dentro do arquivo:** o fluxo para sem escolher um arquivo arbitrário.
- **Title ID/update divergente:** o handoff é interrompido antes do Actions.
- **Hash/MD5 divergente:** a cópia é rejeitada.
- **Workflow falhou:** o helper exibe o link da execução; não declara IPA pronta.
